# 02 TransformData — PalletLogs

Notebook นี้ใช้สำหรับแปลงข้อมูลจากไฟล์ raw ก่อนเข้าสู่ขั้น clean data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

## Load Raw Data

ใช้ไฟล์ raw `PalletLogs.csv` เป็นจุดเริ่มต้น

In [2]:
source_path = '../../data/raw/PalletLogs.csv'
df = pd.read_csv(source_path, encoding='utf-8-sig')

print(f'source: {source_path}')
print(f'shape: {df.shape}')
df.head()

source: ../../data/raw/PalletLogs.csv
shape: (399233, 31)


,PalletID,Action,PalletStatus,PackSize,Amount,LotNo,MaterialCode,NameThai,ProductType,Model,...,FromFloorNo,ToLocationType,ToLocationCode,ToLocationName,ToRowNo,ToColNo,ToFloorNo,UserCreate,UserCreateCode,PlantCode
0,SB1LR260511450,POSTTOCHECKERLOCATION,FOLKPOSTEND,0,0,NaN,NaN,NaN,NaN,NaN,...,0,POSTLOCATION,COM22090001,SB1-1ลานโอน,NaN,0,0,&#3616;&#3588;&#3623;&#3633;&#3605; &#3648;&#3...,2503000002,NaN
1,SB1LR260511451,POSTTOCHECKERLOCATION,FOLKPOSTEND,0,0,NaN,NaN,NaN,NaN,NaN,...,0,POSTLOCATION,COM23110002,ลานจ่าย3 ช่องจ่าย 2 (ครอบ),NaN,0,0,&#3616;&#3588;&#3623;&#3633;&#3605; &#3648;&#3...,2503000002,NaN
2,SB1-1LS3690515P025,POSTTOCHECKERLOCATION,FOLKPOSTEND,320,320,3690515P,ZCB10STDA002000A14,ก/บ คอนกรีตSCGเอลาบานา สีน้ำตาลโอ๊คแดง,CPAC-Tile,CPAC,...,1,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),NaN,0,0,ชัยณรงค์ เพ็ชรนิล,2102000002,NaN
3,SB1LR260511214,POSTTOCHECKERLOCATION,FOLKPOSTEND,0,0,NaN,NaN,NaN,NaN,NaN,...,0,POSTLOCATION,COM23110003,อุปกรณ์,NaN,0,0,เสน่ห์ แสงตระกอน,2202000004,NaN
4,NaN,POSTTOCHECKERLOCATION,NaN,0,0,NaN,NaN,NaN,NaN,NaN,...,0,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),NaN,0,0,ชัยณรงค์ เพ็ชรนิล,2102000002,NaN


## Quick Check in Raw File

ดูค่าที่น่าสงสัยก่อนแปลงข้อมูล

In [3]:
for col in ['Action', 'PalletStatus', 'ProductType', 'FromLocationType', 'ToLocationType']:
    print(f'\n{col}:')
    print(df[col].value_counts(dropna=False).head(10))


Action:
Action
POSTTOCHECKERLOCATION    399233
Name: count, dtype: int64

PalletStatus:
PalletStatus
FOLKPOSTEND      351685
NaN               47286
POSTCOMPLETED       218
GOOD                 42
PACK                  2
Name: count, dtype: int64

ProductType:
ProductType
CPAC-Tile           154473
PRESTIGE-Tile       151380
NaN                  81486
CPAC-Fitting          6781
PRESTIGE-Fitting      3812
ACCESSORIES           1146
DURA-Fitting           155
Name: count, dtype: int64

FromLocationType:
FromLocationType
NORMAL          278397
PALLETSOURCE     90132
BUFFER           30686
FRACTION            17
NaN                  1
Name: count, dtype: int64

ToLocationType:
ToLocationType
POSTLOCATION    399233
Name: count, dtype: int64


## Normalize Missing-like Values

แปลงค่าอย่าง `NULL`, ช่องว่าง, และ `nan` ให้เป็น `NaN` มาตรฐาน

In [4]:
NULL_LIKE_VALUES = ['NULL', '', 'nan', 'NaN', 'None']
df_transform = df.replace(NULL_LIKE_VALUES, np.nan)

df_transform.isna().sum().sort_values(ascending=False).head(15)

Color               399233
ToRowNo             399233
PlantCode           399233
Coat                150673
ProductClass        133839
Curve               120296
FromLocationName     90132
FromLocationCode     90132
FromRowNo            90132
LotNo                89388
Profile              81498
NameThai             81486
Model                81486
ProductType          81486
MaterialCode         81486
dtype: int64

## Drop All-null Columns

ลบคอลัมน์ที่ว่างทั้งหมด (100% missing) เพราะไม่มีประโยชน์ในการวิเคราะห์

In [5]:
all_null_cols = df_transform.columns[df_transform.isna().all()].tolist()
print(f'all-null columns to drop: {all_null_cols}')

df_transform = df_transform.drop(columns=all_null_cols)
print(f'shape after drop: {df_transform.shape}')

all-null columns to drop: ['Color', 'ToRowNo', 'PlantCode']
shape after drop: (399233, 28)


## Convert StampDateTime

แปลง `StampDateTime` จาก string รูปแบบ `DD/MM/YYYY HH:MM:SS` เป็น datetime

In [6]:
df_transform['StampDateTime'] = pd.to_datetime(
    df_transform['StampDateTime'],
    format='%d/%m/%Y %H:%M:%S',
    errors='coerce'
)

print(f'StampDateTime dtype: {df_transform["StampDateTime"].dtype}')
print(f'parse failures: {df_transform["StampDateTime"].isna().sum():,}')
print(f'range: {df_transform["StampDateTime"].min()} → {df_transform["StampDateTime"].max()}')

StampDateTime dtype: datetime64[us]
parse failures: 0
range: 2025-01-02 07:30:08 → 2026-05-18 10:54:28


## Convert Numeric Columns

แปลงคอลัมน์ตำแหน่งและปริมาณให้เป็น numeric

In [7]:
numeric_columns = [
    'PackSize', 'Amount',
    'FromRowNo', 'FromColNo', 'FromFloorNo',
    'ToRowNo', 'ToColNo', 'ToFloorNo',
    'UserCreateCode',
]

for col in numeric_columns:
    if col in df_transform.columns:
        df_transform[col] = pd.to_numeric(df_transform[col], errors='coerce')

numeric_check = pd.DataFrame(
    {
        'column': numeric_columns,
        'non_null': [df_transform[c].notna().sum() if c in df_transform.columns else 0 for c in numeric_columns],
        'missing': [df_transform[c].isna().sum() if c in df_transform.columns else 0 for c in numeric_columns],
    }
)
numeric_check

,column,non_null,missing
0,PackSize,399233,0
1,Amount,399233,0
2,FromRowNo,309101,90132
3,FromColNo,399233,0
4,FromFloorNo,399233,0
5,ToRowNo,0,0
6,ToColNo,399233,0
7,ToFloorNo,399233,0
8,UserCreateCode,399233,0


## Check Converted Data

สรุป dtype หลังแปลง

In [8]:
dtype_summary = pd.DataFrame(
    {
        'column': df_transform.columns,
        'dtype': df_transform.dtypes.astype(str).values,
    }
)

dtype_summary.sort_values(['dtype', 'column']).reset_index(drop=True)

,column,dtype
0,StampDateTime,datetime64[us]
1,FromRowNo,float64
2,Amount,int64
3,FromColNo,int64
4,FromFloorNo,int64
5,PackSize,int64
6,ToColNo,int64
7,ToFloorNo,int64
8,UserCreateCode,int64
9,Action,str


## Save Transformed Data

บันทึกไฟล์ที่แปลงค่าแล้วไว้ใน `data/interim/palletlogs` เพื่อใช้ต่อในขั้น clean

In [9]:
output_path = '../../data/interim/palletlogs/palletlogs_transformed.csv'
df_transform.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'saved: {output_path}')
print(f'shape: {df_transform.shape}')
output_path

saved: ../../data/interim/palletlogs/palletlogs_transformed.csv
shape: (399233, 28)


'../../data/interim/palletlogs/palletlogs_transformed.csv'